# Total precipitation quantiles

In [1]:
import os
import sys
import yaml
from glob import glob
from datetime import datetime, timedelta

import numpy as np
import xarray as xr
import pandas as pd

In [2]:
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
sys.path.insert(0, os.path.realpath('../libs/'))
import verif_utils as vu
import seeps_utils as seeps

In [4]:
config_name = os.path.realpath('verif_config.yml')

with open(config_name, 'r') as stream:
    conf = yaml.safe_load(stream)

In [5]:
model_name = 'IFS'
lead_ind = 4 # day-5

## Verification setup

In [7]:
# ---------------------------------------------------------------------------------------- #
# forecast
filename_OURS = sorted(glob(conf[model_name]['save_loc_gather']+'*.nc'))

# pick years
year_range = conf[model_name]['year_range']
years_pick = np.arange(year_range[0], year_range[1]+1, 1).astype(str)
filename_OURS = [fn for fn in filename_OURS if any(year in fn for year in years_pick)]
filename_OURS = [fn for fn in filename_OURS if '00Z' in fn]

In [8]:
variable_levels = {'total_precipitation': None,}
variable_levels_IFS = {'total_precipitation_6hr': None,}
rename_IFS_to_ERA5 = {'total_precipitation_6hr': 'total_precipitation'}

In [9]:
# ---------------------------------------------------------------------------------------- #
# RMSE compute
for lead_ind in [0, 9]:
    tp_collect = []
    
    for fn_ours in filename_OURS:
        # detect 00Z vs 12Z
        ini = int(fn_ours[-6:-4])
        
        ds_ours = xr.open_dataset(fn_ours)
        ds_ours = vu.ds_subset_everything(ds_ours, variable_levels_IFS)
        ds_ours = ds_ours.rename(rename_IFS_to_ERA5)
        
        ds_ours_24h = vu.accum_6h_24h(ds_ours, ini)
        ds_ours_24h = ds_ours_24h.isel(time=lead_ind)
        
        tp_collect.append(ds_ours_24h)
        
    # Combine verif results
    ds_all_24h = xr.concat(tp_collect, dim='time')
    
    path_verif = conf[model_name]['save_loc_verif']+'Histogram_vals_{:03d}d_{}.nc'.format(
        lead_ind+1, model_name)
    
    ds_all_24h.to_netcdf(path_verif)
    print(path_verif)

/glade/campaign/cisl/aiml/ksha/CREDIT_physics/VERIF/IFS/Histogram_vals_001d_IFS.nc
/glade/campaign/cisl/aiml/ksha/CREDIT_physics/VERIF/IFS/Histogram_vals_010d_IFS.nc


In [12]:
# ds_mean = ds_all_24h['total_precipitation'].mean(dim='time')
# ds_std = ds_all_24h['total_precipitation'].std(dim='time')
# #ds_p95 = ds_all_24h['total_precipitation'].quantile(0.95, dim='time')

# # # Combine these into a new dataset
# ds_summary = xr.Dataset(
#     {
#     'mean': ds_mean,
#     'std': ds_std,
# })

# # Combine these into a new dataset
# # ds_summary = xr.Dataset({'std': ds_std})

In [13]:
# ds_summary.to_netcdf(path_verif)